# Responsive Computation

**Part I · Visualization** — Tutorial 16

Keep the viewer responsive while your script or handler does real work. You
will learn to:

- Flush pending updates before blocking (`flush_async()` / `flush(wait=True)`).
- Offload computation to the user loop (`submit_user` / `run_user` /
  `run_user_sync` / `run_blocking`).
- Use the canonical pattern: show a "Calculating…" banner, await it, run the
  work off-loop, then update the scene and remove the banner.


## Setup


In [ ]:
import asyncio
import time

from pytanga.geometry import Point, Sphere
from pytanga.viz import ControlEvent, VisualizerApp


## 1. Why handlers must not block

Control handlers run on the server's own event loop. A long synchronous
computation inside a handler blocks the loop, so scene updates may never reach
the browser. Use the flush tools and compute offload below.


## 2. Flush tools

- `await flush_async()` — awaitable flush; guarantees pending updates are
  rendered **before** you block.
- `flush(wait=True)` — blocking flush for plain synchronous scripts; it raises a
  clear `RuntimeError` on the server loop.


In [ ]:
viz = VisualizerApp(title="Flush tools")  # reuse the app lifecycle for context

# Inside an async handler:
#   self.viz.set_annotation("Calculating…")
#   await self.viz.flush_async()      # annotation is rendered before blocking
#   result = heavy_sync_work()
#   self.viz.set_annotation(None)

# In a plain synchronous script:
#   viz.flush(wait=True)              # blocking flush (not on the server loop)

print("flush_async() / flush(wait=True) covered")


## 3. Offloading to the user loop

`VisualizerApp` exposes `submit_user()` (fire-and-forget with a one-shot `done=`
callback), `run_user()` (await the result), and `run_user_sync()` (run a plain
sync function). Scripts without a `VisualizerApp` use
`Visualizer.run_blocking(fn)`.


In [ ]:
# submit_user: schedule work off the event loop, then call done(result).
# self.submit_user(_work, done=_done)

# run_user: await the result directly.
# result = await self.run_user(_work)

# run_user_sync: run a plain sync function off-loop.
# result = await self.run_user_sync(time.sleep, 3)

# Without VisualizerApp:
# result = await viz.run_blocking(fn, *args)

print("submit_user / run_user / run_user_sync / run_blocking covered")


## 4. The canonical pattern

Show a "Calculating…" modal banner, await its push, run the work off-loop, then
update the scene and remove the banner in the `done` callback.


In [ ]:
class HeavyWorkApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="Heavy work on release")
        self._sphere_id = "sphere"

    async def init(self) -> None:
        self.viz.add(Sphere(Point(0, 0, 0), radius=1.0), entity_id=self._sphere_id,
                     color="#4488ff", opacity=0.4)
        self.viz.add_slider("radius", label="Radius", min=0.2, max=3.0, step=0.05,
                            value=1.0, on_release=self.on_release)
        self.viz.flush()

    async def on_release(self, value: float, _event: ControlEvent) -> None:
        bid = await self.viz.show_banner_async(
            "## Calculating…\n\nPlease wait.",
            title="Busy",
            dismissable=False,
        )

        async def _work() -> float:
            await asyncio.to_thread(time.sleep, 3)   # simulate blocking compute
            return value

        def _done(result: float) -> None:
            self.viz.update_entity(self._sphere_id, Sphere(Point(0, 0, 0), radius=result))
            self.viz.remove_banner(bid)
            self.viz.flush()

        self.submit_user(_work, done=_done)

# HeavyWorkApp().run()
print("HeavyWorkApp defined")


## Visual Examples

The `heavy_work.py` pattern above: a slider whose release shows a modal banner,
runs a 3 s computation off-loop, then updates the sphere and removes the banner.


## Summary

| Task | API |
|---|---|
| Awaitable flush | `await flush_async()` |
| Blocking flush (scripts) | `flush(wait=True)` (raises on the server loop) |
| Fire-and-forget offload | `submit_user(work, done=callback)` |
| Await offload | `await run_user(work)` |
| Sync offload | `await run_user_sync(fn)` |
| No-app offload | `await viz.run_blocking(fn)` |

**Next:** [17 — Export](../17_export/).
